# M33 lab solution: Follow-up test

**Brief.** Write 15 three-turn conversations and compare retrieval hit rate on turns two and three with and without history-aware query rewriting.

`../data/conversations.jsonl` holds 15 three-turn conversations about the Larkspur handbook; every turn has evidence spans, so each retrieval is scored with `lab_kit.is_hit`. Turns two and three are follow-ups ("And on Enterprise?") that name their topic only through the turns before them. The rewriting step is the contextualise prompt of `create_history_aware_retriever` from lesson 33.4, written out so the rewritten queries can be printed.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## The Part V system, rebuilt inline

This lab does not depend on another lab's output, so the cell below rebuilds the minimal Part V system: the handbook loaded with `lab_kit`, the baseline chunker (500 characters, 50 overlap), an in-memory vector store over `get_embeddings()`, and an answer prompt that cites passages by number. Later cells reuse `retriever`, `format_docs` and `rag_answer`.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

handbook = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
embeddings = get_embeddings()
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook. Use only the numbered passages. "
               "Cite each claim with its passage number in square brackets, like [2]. "
               "If the passages do not contain the answer, reply: The handbook does not say."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(docs, 1))


def rag_answer(question, docs=None):
    """Retrieve (unless docs are given) and answer with citations; returns (answer, docs)."""
    docs = retriever.invoke(question) if docs is None else docs
    return answer_chain.invoke({"context": format_docs(docs), "question": question}), docs


print(len(chunks), "chunks;", len(golden), "golden questions")

In [ ]:
conversations = load_jsonl(DATA / "conversations.jsonl")
print(len(conversations), "conversations;", sum(len(c["turns"]) for c in conversations), "turns")

## 1. History-aware query rewriting

`create_history_aware_retriever(llm, retriever, prompt)` sends the question straight to the retriever when the history is empty, and otherwise runs `prompt | llm | StrOutputParser()` and retrieves with the result. The function below does the same and also returns the query, so the log can show it. The history holds the user's earlier turns.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import MessagesPlaceholder

CONTEXTUALIZE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and the latest user question, which might refer to the history, "
               "write a standalone question that can be understood without the history. "
               "Do not answer it; only reformulate it if needed, otherwise return it as is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
contextualize = CONTEXTUALIZE_PROMPT | llm | StrOutputParser()


def history_aware_search(question, history):
    query = contextualize.invoke({"input": question, "chat_history": history}) if history else question
    return query, retriever.invoke(query)

## 2. Every turn, with and without rewriting

In [ ]:
rows = []
for conv in conversations:
    history = []
    for t in conv["turns"]:
        plain_docs = retriever.invoke(t["question"])
        query, aware_docs = history_aware_search(t["question"], history)
        rows.append({"conversation": conv["id"], "turn": t["turn"], "question": t["question"], "rewritten": query,
                     "hit without rewriting": any(is_hit(d.page_content, t["evidence_spans"]) for d in plain_docs),
                     "hit with rewriting": any(is_hit(d.page_content, t["evidence_spans"]) for d in aware_docs)})
        history.append(HumanMessage(t["question"]))
log = pd.DataFrame(rows)
print(log[log.turn > 1][["conversation", "turn", "question", "rewritten", "hit without rewriting", "hit with rewriting"]]
      .to_string(index=False))

## 3. Results: hit rate at k=4 by turn

In [ ]:
summary = log.groupby("turn")[["hit without rewriting", "hit with rewriting"]].mean().round(2)
summary.insert(0, "turns", log.groupby("turn").size())
follow_ups = log[log.turn > 1]
summary.loc["2 and 3"] = [len(follow_ups), round(follow_ups["hit without rewriting"].mean(), 2),
                          round(follow_ups["hit with rewriting"].mean(), 2)]
print(summary.to_string())

## What this shows

- The "2 and 3" row is the brief's comparison: hit rate on follow-up turns with and without the rewrite. Turn 1 has no history, so both columns are the same retrieval there and serve as a check.
- A follow-up such as "And on Enterprise?" carries almost no retrievable words of its own, which is why the plain retriever misses on it. The rewritten column shows what the model added from the history.
- The rewrite is one extra model call on every turn after the first. A rewrite that drifts (adds a topic the user did not ask about) shows up here as a miss in the "with" column; read those rows before trusting the gain.